In [9]:
import numpy as np
import pandas as pd

In [10]:
rng = np.random.default_rng(seed=42)
N_PATIENTS = 2000

In [11]:
print("Pronto! numpy", np.__version__, "| pandas", pd.__version__)

Pronto! numpy 2.5.4 | pandas 3.1.1


In [12]:
CLASS_PROPORTIONS = {
    "nociceptive": 0.40,
    "neuropathic": 0.15,
    "nociplastic": 0.20,
    "mixed": 0.25,
}

labels = rng.choice(
    list(CLASS_PROPORTIONS.keys()),
    size=N_PATIENTS,
    p=list(CLASS_PROPORTIONS.values()),
)

df = pd.DataFrame({
    "patient_id": range(1, N_PATIENTS + 1),
    "mechanism": labels,
})

df["mechanism"].value_counts(normalize=True).round(3)

mechanism
nociceptive    0.394
mixed          0.262
nociplastic    0.191
neuropathic    0.152
Name: proportion, dtype: float64

In [13]:
DURATION_PARAMS = {
    "nociceptive": (2, 0.9),
    "neuropathic": (6, 0.8),
    "nociplastic": (24, 0.7),
    "mixed": (12, 0.8),
}

duration = np.empty(N_PATIENTS)

for mechanism, (median, spread) in DURATION_PARAMS.items():
    mask = df["mechanism"] == mechanism
    duration[mask] = rng.lognormal(mean=np.log(median), sigma=spread, size=mask.sum())

df["pain_duration_months"] = np.clip(duration, 0.25, 240).round(1)

is_nociplastic = df["mechanism"] == "nociplastic"
df.loc[is_nociplastic, "pain_duration_months"] = df.loc[is_nociplastic, "pain_duration_months"].clip(lower=3)

df.groupby("mechanism")["pain_duration_months"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
mechanism,,,,,,,,
mixed,524.0,16.9,15.9,1.5,6.9,12.1,20.4,116.9
neuropathic,305.0,8.8,8.4,0.7,3.6,6.1,11.1,69.4
nociceptive,789.0,3.0,3.3,0.2,1.1,2.1,3.6,26.2
nociplastic,382.0,30.1,25.0,3.5,15.0,24.1,37.2,208.5


In [14]:
DISTRIBUTION_PROBS = {
    "nociceptive": [0.75, 0.05, 0.18, 0.02],
    "neuropathic": [0.10, 0.70, 0.15, 0.05],
    "nociplastic": [0.05, 0.05, 0.35, 0.55],
    "mixed":       [0.25, 0.30, 0.30, 0.15],
}
DISTRIBUTION_CATEGORIES = ["localized", "dermatomal", "regional", "widespread"]

distribution = np.empty(N_PATIENTS, dtype=object)

for mechanism, probs in DISTRIBUTION_PROBS.items():
    mask = df["mechanism"] == mechanism
    distribution[mask] = rng.choice(DISTRIBUTION_CATEGORIES, size=mask.sum(), p=probs)

df["pain_distribution"] = distribution

pd.crosstab(df["mechanism"], df["pain_distribution"], normalize="index").round(2)

pain_distribution,dermatomal,localized,regional,widespread
mechanism,,,,
mixed,0.27,0.25,0.33,0.15
neuropathic,0.78,0.06,0.14,0.03
nociceptive,0.05,0.77,0.16,0.02
nociplastic,0.05,0.06,0.41,0.49


In [15]:
# Questionnaire scores per mechanism: (mean, standard deviation)
DN4_PARAMS = {
    "nociceptive": (1.5, 1.2),
    "neuropathic": (5.5, 1.8),
    "nociplastic": (3.0, 1.5),
    "mixed":       (4.0, 1.8),
}
PAINDETECT_PARAMS = {
    "nociceptive": (8, 4),
    "neuropathic": (22, 5),
    "nociplastic": (15, 5),
    "mixed":       (16, 6),
}

# Empty arrays for the scores
dn4 = np.empty(N_PATIENTS)
paindetect = np.empty(N_PATIENTS)

# Sample scores for each mechanism
for mechanism in CLASS_PROPORTIONS:
    mask = df["mechanism"] == mechanism
    n = mask.sum()

    mean, sd = DN4_PARAMS[mechanism]
    dn4[mask] = rng.normal(loc=mean, scale=sd, size=n)

    mean, sd = PAINDETECT_PARAMS[mechanism]
    paindetect[mask] = rng.normal(loc=mean, scale=sd, size=n)

# Keep scores within each scale and as whole numbers
df["dn4_score"] = np.clip(dn4, 0, 10).round().astype(int)
df["paindetect_score"] = np.clip(paindetect, -1, 38).round().astype(int)

# Check: mean score and % above the cut-off
df.groupby("mechanism").agg(
    dn4_mean=("dn4_score", "mean"),
    dn4_positive=("dn4_score", lambda s: (s >= 4).mean()),
    paindetect_mean=("paindetect_score", "mean"),
    paindetect_positive=("paindetect_score", lambda s: (s >= 19).mean()),
).round(2)

,dn4_mean,dn4_positive,paindetect_mean,paindetect_positive
mechanism,,,,
mixed,3.89,0.56,15.79,0.34
neuropathic,5.29,0.85,22.11,0.74
nociceptive,1.56,0.06,8.03,0.00
nociplastic,3.03,0.38,14.96,0.24
